
# Create Metadata Files

This notebook manages `metadata.json` and `private_metadata.json`.

- `metadata.json` is the public pointer file: which dataset is analysed, which mode applies, and which analysis parameters are used. It is tracked by git.
- `private_metadata.json` contains student-specific information. It is ignored by git and stays on your machine.

**First decide in Section 0 how the notebook should behave:** keep the existing files, rebuild them completely from the parameter cells in this notebook, or update the existing files only where this notebook sets a value.

All settings are variables in the parameter cells below, filled with working example values. The comment next to each value explains what it means and when you have to change it.



## Section 0: Choose What This Notebook Does

Decide how to treat the metadata files **before running anything else**:

| Mode | Behaviour |
|------|-----------|
| `1` | **Load existing files.** Nothing is written. Use this when your `metadata.json` and `private_metadata.json` are already correct and you only want to check them. |
| `2` | **Overwrite completely.** Both files are rebuilt from the parameter cells in this notebook. Manual edits in the JSON files are lost. |
| `3` | **Update existing files.** Only values that are **not `None`** in this notebook overwrite the existing files. Set a parameter to `None` to keep whatever is currently in the file. |

**Warning:** modes `2` and `3` overwrite the files when the write cells run. If you are unsure, start with mode `1` and look at the output of Section 6 first.


In [ ]:
# Section 0: Metadata Mode
#   1 = load existing files, write nothing
#   2 = overwrite both files completely with the values from this notebook
#   3 = update existing files, only non-None values from this notebook are written
metadata_mode = 1

if metadata_mode not in (1, 2, 3):
    raise ValueError('metadata_mode must be 1 (load existing), 2 (overwrite), or 3 (update non-None values)')

mode_descriptions = {
    1: 'Load existing metadata files - nothing will be written.',
    2: 'Overwrite metadata.json and private_metadata.json completely.',
    3: 'Update existing files with the non-None values from this notebook.',
}
print(f'metadata_mode = {metadata_mode}: {mode_descriptions[metadata_mode]}')


## Section 1: Import Helpers

This cell is already prepared for you. Run it once before changing anything else.

In [ ]:
# Section 1: Import Helpers
from pathlib import Path
import json
import sys

project_root = Path.cwd()
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from metadata_loader import (
    load_metadata_context,
    load_private_metadata,
    load_public_metadata,
    merge_metadata_updates,
    save_private_metadata,
    save_public_metadata,
    summarize_metadata_context,
)

print('Helpers imported successfully.')



## Section 2: Choose the Dataset

*Skipped in mode `1` - the existing `metadata.json` already contains these values.*

Set `recorded_data_path` to the measurement file you want to analyse. This choice selects the use case:

- `data/drivetrain/Example/Raw Data.csv` (drivetrain, illuminance)
- `data/suspension/Example/Beschleunigung ohne g.xls` (suspension, acceleration)

Then fill in the general metadata about this measurement run. Keep `measurement_type`, `quantity`, and `recorded_data_path` consistent: they decide which analysis mode the evaluation notebook runs.


In [ ]:
# Section 2: Dataset and General Metadata
# In mode 3, set any of these to None to keep the value from the existing metadata.json.

# Path to the measurement file, relative to the project root.
# CHANGE THIS when you record your own data or switch the use case:
#   drivetrain example: 'data/drivetrain/Example/Raw Data.csv'
#   suspension example: 'data/suspension/Example/Beschleunigung ohne g.xls'
recorded_data_path = 'data/suspension/Example/Beschleunigung ohne g.xls'

# Use case of the measurement: 'drivetrain' (light sensor) or 'suspension' (acceleration sensor).
# Must match the folder in recorded_data_path.
measurement_type = 'suspension'

# Name of this measurement run. Use a short, recognisable name for your own runs,
# e.g. 'Run_01_gravel' or your group name. The example data uses 'Example'.
run_name = 'Example'

# Physical quantity recorded in the file: 'illuminance' for drivetrain, 'acceleration' for suspension.
quantity = 'acceleration'

# Processing stage of the data this file points to. Keep 'raw' for unprocessed sensor exports.
data_stage = 'raw'

# Version of your metadata/analysis setup. Increase it (e.g. 'v0.2.0') when you
# change parameters and want to distinguish the results.
version = 'v0.1.0'

# Optional path to a fast local working copy of the data (leave '' if unused).
hot_storage_path = ''

if metadata_mode == 1:
    print('metadata_mode = 1: the values above are ignored, the existing metadata.json is used.')
else:
    if recorded_data_path is not None:
        data_file = project_root / recorded_data_path
        if not data_file.exists():
            raise FileNotFoundError(f'Recorded data file not found: {data_file}')
    print('Selected dataset:', recorded_data_path)
    print('Use case (measurement_type):', measurement_type)
    print('Run name:', run_name)
    print('Quantity:', quantity)



## Section 3: Use-Case Settings

*Skipped in mode `1` - the existing `metadata.json` already contains these values.*

The next two cells hold all settings for the two use cases. **Fill in the cell that matches your use case** (printed above). Run both cells anyway: `metadata.json` always contains both blocks, so you can switch datasets later without recreating the file.

In mode `3`, set any value to `None` to keep what is currently in `metadata.json`.

Each use case has two groups of settings:

1. **Analysis settings** - which columns are analysed, which time range is used, and how smoothing and outlier detection behave.
2. **Setup description** - facts about the physical experiment (gears and rotor marker for drivetrain, sensor axes for suspension).


### Use Case A: Drivetrain (illuminance)

The light sensor sees bright/dark cycles from a marker on the rotor. From these cycles the notebook derives rotor and motor speed, using the gear teeth counts below.

In [ ]:
# Section 3A: Drivetrain Analysis Settings

# Column names as they appear in the CSV file. If a name does not exist in the
# loaded file, the analysis falls back to automatic detection.
drivetrain_time_column = 'Time (s)'          # column with the time stamps
drivetrain_value_column = 'Illuminance (lx)' # column with the light-sensor signal

# Time range to analyse, in seconds. None = use the full recording.
# Set numbers (e.g. 2.0 and 15.0) to cut off shaky starts or ends.
drivetrain_analysis_start_s = None
drivetrain_analysis_end_s = None

# Number of neighbouring table rows averaged for the smoothed curve.
# Larger = smoother, but short bright/dark transitions can get lost.
drivetrain_smoothing_window = 5

# Z-score above which a raw illuminance value is marked as a possible outlier.
# Lower = stricter (more points marked), higher = more tolerant.
drivetrain_outlier_z_threshold = 3.0

# Z-score for outliers in the derived motor speed (rpm).
drivetrain_motor_speed_outlier_z_threshold = 3.0

# Smoothing windows compared side by side in the parameter study.
drivetrain_motor_speed_smoothing_windows = [1, 5, 15, 31]

# Show the raw and/or smoothed curve in the measurement plot.
drivetrain_plot_raw_values = True
drivetrain_plot_smoothed_values = True

# Section 3A: Drivetrain Setup Description
# Describe YOUR physical setup here - these values go into the speed calculation.

# How many bright/dark cycles the rotor marker produces per full rotor rotation.
# One reflective stripe on the rotor = 1. Two stripes = 2.
bright_dark_cycles_per_rotation = 1

# Position of the gear switch during YOUR measurement: 'towards motor' or 'towards rotor'.
first_gear_switch_position = 'towards motor'

# Teeth counts of the first gear stage for both switch positions.
# Count the teeth on your car if they differ from the example.
first_gear_settings = {
    'towards motor': {'motor_gear_teeth': 12, 'rotor_gear_teeth': 36},
    'towards rotor': {'motor_gear_teeth': 12, 'rotor_gear_teeth': 12},
}

# Further gear stages between motor and rotor. Add or remove entries to match your car.
gear_combos = [
    {'name': 'gear combo 2', 'motor_gear_teeth': 12, 'rotor_gear_teeth': 36},
    {'name': 'gear combo 3', 'motor_gear_teeth': 12, 'rotor_gear_teeth': 36},
]

print('Drivetrain settings defined.')


### Use Case B: Suspension (acceleration)

The phone records linear acceleration on three axes. The main axis is used to derive vehicle speed; all three axes are shown as acceleration and G-force plots. The column names must match the recorded file exactly.

In [ ]:
# Section 3B: Suspension Analysis Settings

# Column names as they appear in the Excel export. If a name does not exist in
# the loaded file, the analysis falls back to automatic detection.
# Check them with your own export - the phone app must be oriented the same way.
suspension_time_column = 'Time (s)'                            # column with the time stamps
suspension_value_column = 'Absolute acceleration (m/s^2)'      # primary signal for quality checks and outliers
main_axis_column = 'Linear Acceleration x (m/s^2)'             # main driving direction (used to derive vehicle speed)
lateral_axis_column = 'Linear Acceleration y (m/s^2)'          # sideways axis
vertical_axis_column = 'Linear Acceleration z (m/s^2)'         # vertical axis

# Time range to analyse, in seconds. None = use the full recording.
# Set numbers (e.g. 1.5 and 20.0) to cut off the handling before/after the run.
suspension_analysis_start_s = None
suspension_analysis_end_s = None

# Number of neighbouring table rows averaged for the smoothed curve.
# Acceleration data is noisy, so this is much larger than for drivetrain.
suspension_smoothing_window = 100

# Z-score above which a raw acceleration value is marked as a possible outlier.
suspension_outlier_z_threshold = 3.0

# Z-score for outliers in the derived speed and G-force curves.
# Increase it if too many points are red, lower it if obvious spikes are not marked.
motion_outlier_z_threshold = 7.5

# Vehicle speed in m/s at the start of the recording. 0.0 = car starts from standstill.
speed_initial_m_per_s = 0.0

# Smoothing windows compared side by side in the parameter study.
parameter_smoothing_windows = [5, 25, 75, 151]

# Show the raw and/or smoothed curves in the measurement plots.
suspension_plot_raw_values = True
suspension_plot_smoothed_values = True

# Section 3B: Suspension Setup Description
# Describe YOUR sensor setup here - it documents how the axes have to be read.

# Unit of the acceleration columns (as exported by the phone app).
acceleration_unit = 'm/s^2'

# Which physical direction each configured axis corresponds to on your car.
# Adjust these if your phone was mounted in a different orientation.
speed_axis_description = 'main vehicle acceleration direction'
lateral_axis_description = 'sideways acceleration'
vertical_axis_description = 'vertical acceleration'

print('Suspension settings defined.')



## Section 4: Write `metadata.json`

What happens here depends on the mode from Section 0:

- Mode `1`: nothing is written, the existing file is loaded and shown.
- Mode `2`: the file is rebuilt from the parameters above - nothing else is kept.
- Mode `3`: the existing file is loaded and every parameter above that is not `None` overwrites the stored value.

Check the printed JSON: this is exactly what the analysis notebook will use.


In [ ]:
# Section 4: Assemble and Write metadata.json
if metadata_mode == 1:
    public_metadata = load_public_metadata(project_root)
    print('metadata_mode = 1: loaded existing metadata.json, nothing written.')
else:
    analysis_metadata = {
        'drivetrain_illuminance': {
            'time_column': drivetrain_time_column,
            'value_column': drivetrain_value_column,
            'analysis_start_s': drivetrain_analysis_start_s,
            'analysis_end_s': drivetrain_analysis_end_s,
            'smoothing_window': drivetrain_smoothing_window,
            'outlier_z_threshold': drivetrain_outlier_z_threshold,
            'motor_speed_outlier_z_threshold': drivetrain_motor_speed_outlier_z_threshold,
            'motor_speed_smoothing_windows': drivetrain_motor_speed_smoothing_windows,
            'plot_raw_values': drivetrain_plot_raw_values,
            'plot_smoothed_values': drivetrain_plot_smoothed_values,
        },
        'suspension_acceleration': {
            'time_column': suspension_time_column,
            'value_column': suspension_value_column,
            'main_axis_column': main_axis_column,
            'lateral_axis_column': lateral_axis_column,
            'vertical_axis_column': vertical_axis_column,
            'analysis_start_s': suspension_analysis_start_s,
            'analysis_end_s': suspension_analysis_end_s,
            'smoothing_window': suspension_smoothing_window,
            'outlier_z_threshold': suspension_outlier_z_threshold,
            'motion_outlier_z_threshold': motion_outlier_z_threshold,
            'speed_initial_m_per_s': speed_initial_m_per_s,
            'parameter_smoothing_windows': parameter_smoothing_windows,
            'plot_raw_values': suspension_plot_raw_values,
            'plot_smoothed_values': suspension_plot_smoothed_values,
        },
    }

    suspension_metadata = {
        'acceleration_unit': acceleration_unit,
        'speed_axis_description': speed_axis_description,
        'lateral_axis_description': lateral_axis_description,
        'vertical_axis_description': vertical_axis_description,
    }

    drivetrain_metadata = {
        'rotor_marker': {
            'bright_dark_cycles_per_rotation': bright_dark_cycles_per_rotation,
        },
        'first_gear_combo': {
            'switch_position': first_gear_switch_position,
            'settings': first_gear_settings,
        },
        'gear_combos': gear_combos,
    }

    notebook_metadata = {
        'recorded_data_path': recorded_data_path,
        'measurement_type': measurement_type,
        'run_name': run_name,
        'quantity': quantity,
        'data_stage': data_stage,
        'version': version,
        'hot_storage_path': hot_storage_path,
        'analysis': analysis_metadata,
        'suspension': suspension_metadata,
        'drivetrain': drivetrain_metadata,
    }

    if metadata_mode == 3:
        public_metadata = merge_metadata_updates(load_public_metadata(project_root), notebook_metadata)
    else:
        public_metadata = notebook_metadata

    public_path = save_public_metadata(public_metadata, project_root)
    print('Wrote', public_path)

print(json.dumps(public_metadata, indent=2))



## Section 5: Private Metadata

Enter your name. `private_metadata.json` is ignored by git, so this information stays on your machine.

The mode from Section 0 applies here too: mode `1` loads the existing file, mode `2` overwrites it, mode `3` updates only non-`None` values.


In [ ]:
# Section 5: Write private_metadata.json

# Replace the example values with your own name.
# In mode 3, set a value to None to keep the name from the existing file.
student_first_name = 'Vorname'
student_last_name = 'Name'

if metadata_mode == 1:
    private_metadata = load_private_metadata(project_root)
    print('metadata_mode = 1: loaded existing private_metadata.json, nothing written.')
else:
    notebook_private_metadata = {
        'student': {
            'first_name': student_first_name,
            'last_name': student_last_name,
        },
    }

    if metadata_mode == 3:
        private_metadata = merge_metadata_updates(load_private_metadata(project_root), notebook_private_metadata)
    else:
        private_metadata = notebook_private_metadata

    private_path = save_private_metadata(private_metadata, project_root)
    print('Wrote', private_path)

print(json.dumps(private_metadata, indent=2))


## Section 6: Verify

Reload both files the same way the analysis notebook does and check that the resolved data path exists. If this cell runs without errors, you can continue with `lab_06_evaluate_measurements_jupyter.ipynb`.

In [ ]:
# Section 6: Verify the Generated Files
metadata_context = load_metadata_context(project_root)
summary = summarize_metadata_context(metadata_context)

selected_data_path = metadata_context['selected_data_path']
print('Selected data path exists:', selected_data_path.exists())
print(json.dumps({key: value for key, value in summary.items() if key not in ['analysis', 'suspension', 'drivetrain']}, indent=2))